# m16 lab · Magnetic localization

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A magnetic localization system drives known coils and reads the voltages they induce in small sensor coils in the catheter. You compute the dipole field and the coil voltage, solve for a sensor's position and direction by nonlinear least squares, predict its precision, see why one coil gives only five degrees of freedom, and measure the bias that a nearby conductor adds.

| Part | What you do | Equation |
| --- | --- | --- |
| A | The dipole field and the coil voltage | eq-m16-dipole, eq-m16-emf |
| B | Solving for the pose | eq-m16-coupling, eq-m16-crlb |
| C | Precision against distance | eq-m16-sensitivity |
| D | Five degrees of freedom, and six | eq-m16-5dof |
| E | Distortion from a nearby conductor | eq-m16-distortion |

Lengths in mm; fields in relative units with mu0/(4 pi) = 1. The generator layout (three coil trios on a 150 mm circle), the noise levels and the conductor are illustrative, not any product's design. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import magnetic as M

## Part A · The dipole field and the coil voltage

A small coil carrying current is a magnetic dipole $\mathbf m$; its field is
$(3(\mathbf m\cdot\hat{\mathbf r})\hat{\mathbf r} - \mathbf m)/r^3$ (eq-m16-dipole), falling as the cube of distance. A
sensor coil of $N$ turns and area $A$ in a field oscillating at $f$ sees a voltage of amplitude $2\pi f N A B_n$
(eq-m16-emf).

In [ ]:
m = np.array([0.0, 0.0, 1e6])
r = np.logspace(1.7, 2.8, 60)
on_axis = np.array([np.linalg.norm(M.dipole_field(m, np.array([0, 0, ri]))) for ri in r])
side = np.array([np.linalg.norm(M.dipole_field(m, np.array([ri, 0, 0]))) for ri in r])
plt.loglog(r, on_axis, label="on the axis"); plt.loglog(r, side, label="in the equatorial plane")
plt.xlabel("distance (mm)"); plt.ylabel("|B| (relative)"); plt.legend(); plt.show()
slope = np.polyfit(np.log(r), np.log(on_axis), 1)[0]
print(f"log-log slope {slope:.3f}; axis / equator = {on_axis[0] / side[0]:.3f}")
print("coil voltage, 100 turns, 1 mm², 10 kHz, 1 µT normal field:", M.emf_amplitude(10e3, 100, 1e-6, 1e-6) * 1e6, "µV")

In [ ]:
# Checks
assert abs(slope + 3) < 1e-9
assert abs(on_axis[0] / side[0] - 2) < 1e-12

## Part B · Solving for the pose

Each generator coil $k$ makes a field $\mathbf B_k(\mathbf p)$ at the sensor; a single sensor coil with axis
$\hat{\mathbf n}$ reads $V_k = \hat{\mathbf n}\cdot\mathbf B_k(\mathbf p)$ (eq-m16-coupling). Nine generator coils give
nine readings for five unknowns (three of position, two of direction). Solve by nonlinear least squares from noisy
readings, and compare the spread with the prediction $\sigma^2(J^\top J)^{-1}$ (eq-m16-crlb).

In [ ]:
gen = M.generator()
true = np.array([10.0, -20.0, 150.0, 0.6, 1.1])
v = M.readings(true, gen)
rng = np.random.default_rng(16)
sigma = 1e-4
ests = np.array([M.solve_pose(v + sigma * rng.standard_normal(v.size), gen, true + [5, -5, 5, 0.05, -0.05]) for _ in range(500)])
mc = np.sqrt(np.sum(ests[:, :3].var(axis=0)))
print(f"Monte Carlo RMS position error {mc:.4f} mm; predicted {M.position_sd(true, gen, sigma):.4f} mm")
print("mean error (mm):", np.round(ests[:, :3].mean(axis=0) - true[:3], 4))

In [ ]:
# Checks
assert abs(mc / M.position_sd(true, gen, sigma) - 1) < 0.1
assert np.all(np.abs(ests[:, :3].mean(axis=0) - true[:3]) < 0.02)

## Part C · Precision against distance

On a dipole's axis $|B| = 2m/r^3$, so a field error $\delta B$ moves the range by $r\,\delta B/(3|B|)$
(eq-m16-sensitivity): at fixed noise, the range error grows as $r^4$. Far from the generator the full pose does at least as
badly, because the coils also look closer together; near the coils, where distances to them differ a lot, it grows more
slowly.

In [ ]:
z = np.logspace(2, 2.8, 30)
sd = np.array([M.position_sd([0.0, 0.0, zi, 0.6, 1.1], gen, 1e-4) for zi in z])
plt.loglog(z, sd, label="full pose, predicted")
plt.loglog(z, sd[0] * (z / z[0]) ** 4, "--", c="gray", label="slope 4")
plt.xlabel("height above the generator (mm)"); plt.ylabel("RMS position error (mm)"); plt.legend(); plt.show()
local = np.diff(np.log(sd)) / np.diff(np.log(z))
print("local slopes from", round(local[0], 2), "to", round(local[-1], 2))

In [ ]:
# Checks
assert np.all(np.diff(local) > 0) and local[-1] > 4.0
assert abs(M.axial_range_error(200, 2e6 / 200**3, 1e-3) / M.axial_range_error(100, 2e6 / 100**3, 1e-3) - 16) < 1e-9

## Part D · Five degrees of freedom, and six

Turning a single coil about its own axis changes no reading (eq-m16-5dof), so one coil gives position and direction,
five degrees of freedom. Two coils at an angle, fixed to each other, reveal the roll too.

In [ ]:
p, n1 = true[:3], M.axis_from_angles(0.6, 1.1)
n2 = M.axis_from_angles(0.6 + np.pi / 2, 1.1)              # perpendicular to n1
def rank_two_coils(p, n1, n2, h=1e-5):
    cols = []
    G = lambda q: M.field_matrix(q, gen)
    for j in range(3):
        e = np.zeros(3); e[j] = h
        cols.append(np.concatenate([G(p + e) @ n1 - G(p - e) @ n1, G(p + e) @ n2 - G(p - e) @ n2]) / (2 * h))
    for ax in np.eye(3):
        cols.append(np.concatenate([G(p) @ (M.rotate(n1, ax, h) - M.rotate(n1, ax, -h)),
                                    G(p) @ (M.rotate(n2, ax, h) - M.rotate(n2, ax, -h))]) / (2 * h))
    s = np.linalg.svd(np.stack(cols, axis=1), compute_uv=False)
    return int(np.sum(s > 1e-8 * s[0]))
print("one coil: rank", M.rotation_jacobian_rank(p, n1, gen), "of 6")
print("two coils: rank", rank_two_coils(p, n1, n2), "of 6")

In [ ]:
# Checks
assert M.rotation_jacobian_rank(p, n1, gen) == 5 and rank_two_coils(p, n1, n2) == 6

## Part E · Distortion from a nearby conductor

A conductor near the field responds with eddy currents: model it as an induced dipole $\alpha\mathbf B_k(\mathbf q)$ at
$\mathbf q$ for each generator coil (eq-m16-distortion). The solver does not know it is there, so the pose is biased.
Compare the first-order bias $(J^\top J)^{-1}J^\top\delta\mathbf V$ with a full solve as the conductor moves away.

In [ ]:
rows = []
for hq in (200.0, 250.0, 300.0, 400.0):
    q = np.array([60.0, 0.0, hq])
    dv = M.distorted_readings(true, gen, q, 2e3) - v
    full = M.solve_pose(v + dv, gen, true)[:3] - true[:3]
    lin = M.linear_bias(true, gen, dv)[:3]
    rows.append((hq, np.linalg.norm(full), np.linalg.norm(lin)))
    print(f"conductor at height {hq:.0f} mm: bias {np.linalg.norm(full):.3f} mm (first order {np.linalg.norm(lin):.3f} mm)")

In [ ]:
# Checks
assert all(a[1] > b[1] for a, b in zip(rows, rows[1:]))
assert all(abs(r[1] - r[2]) < 0.05 * r[1] + 1e-3 for r in rows)

## Where to go next

- Add a second sensor coil to the solver and estimate all six degrees of freedom.
- Make $\alpha$ complex (a phase lag, as eddy currents have) and see the readings' phase reveal the distortion.
- m17 locates electrodes from currents instead of fields; m18 combines the two.